# Kiểm chứng metadata S3 v1

Mặc định chỉ preflight, không gọi mạng. Chạy tuần tự trên một runtime; không mở hai phiên ghi cùng output. Kết quả nằm riêng trên Drive. Timestamp bundle thiếu múi giờ giữ ở inconclusive. Metadata phù hợp không chứng minh nội dung S3 là v1.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Chuẩn bị runtime
Upload `recovery-runtime.zip` vào `MyDrive/recovery-colab-kit/`, rồi chạy cell bên dưới. Gói chỉ chứa mã nguồn và journal; không cần corpus M1.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, hashlib, json

# Upload recovery-runtime.zip vào thư mục này trên Drive.
KIT = Path("/content/drive/MyDrive/recovery-colab-kit")
ARCHIVE = KIT / "recovery-runtime.zip"
assert ARCHIVE.is_file(), f"Chưa thấy {ARCHIVE}. Upload recovery-runtime.zip vào KIT hoặc sửa KIT."
REPO = Path("/content/recovery-runtime")
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as bundle:
    for member in bundle.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError("Unsafe archive path")
    bundle.extractall(REPO)
manifest = json.loads((REPO / "runtime-manifest.json").read_text())
for name, expected in manifest["sha256"].items():
    with (REPO / name).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    assert actual == expected, f"Hash mismatch: {name}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-recovery.txt")])
sys.path.insert(0, str(REPO))
print("Runtime và 5 journal đã sẵn sàng:", REPO)


In [ ]:
from src.benchmark.recovery_review import read_journal, unavailable_by_run, select_s3_sample, review_metadata, retry_unavailable
BASE = REPO / "evidence"
JOURNALS = {
    "only-v1-drive-evidence": BASE / "only-v1-drive-evidence/work/versions.jsonl",
    "multi-dang-0000-4999-drive-evidence": BASE / "multi-dang-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-hieu-0000-4999-drive-evidence": BASE / "multi-hieu-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-thien-0000-4999-drive-evidence": BASE / "multi-thien-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-tien-0000-4999-drive-evidence": BASE / "multi-tien-0000-4999-drive-evidence/work/versions.jsonl",
}
OUTPUT = KIT / "results" / "metadata"
RUN_NETWORK = False  # Đổi thành True sau khi preflight chạy thành công.
assert len(JOURNALS) == 5 and len(set(p.resolve() for p in JOURNALS.values())) == 5
states = {name: read_journal(p) for name,p in JOURNALS.items()}
print("Records:", {name:len(s) for name,s in states.items()})
print("Unavailable pairs:", len(unavailable_by_run(states)))
sample = select_s3_sample(states["only-v1-drive-evidence"].values(), count=1000, seed=42)
print("S3 sample:", len(sample), "unique IDs:", len({r["arxiv_id"] for r in sample}))
print("Output:", OUTPUT)


In [ ]:
if RUN_NETWORK:
    summary = review_metadata(JOURNALS, OUTPUT, sample_run='only-v1-drive-evidence', count=1000, seed=42)
    print(summary)
else:
    print("Preflight only. Đặt RUN_NETWORK=True để thực thi.")

## Kết quả và chạy lại
Kết quả nằm ở `KIT/results/metadata/`. Retry có thể resume khi chạy lại notebook với cùng gói đầu vào và OUTPUT; các file đã tải được kiểm tra hash. Metadata chạy lại sẽ gọi discovery mới cho toàn bộ mẫu. Chạy tuần tự hai notebook để các request cùng tuân thủ limiter; không mở hai runtime crawl song song. Nếu đổi gói đầu vào, hãy dùng OUTPUT mới. Không sửa manifest.
Nếu Colab ngắt giữa một dòng journal và lần sau báo lỗi journal, giữ nguyên output để kiểm tra trước khi chạy tiếp. Timestamp bundle thiếu múi giờ vẫn là inconclusive.